# Track Vision: 01_Training

## Import Dependencies

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import onnx
import onnxsim
from IPython.display import display, clear_output
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using execution device: {device}")

## Define Constants

In [ ]:
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

## Augment Images

Apply random crops, flips, and color adjustments to prevent model overfitting.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Build Pipeline

Configure PyTorch image folders and data loaders for batch processing.

In [ ]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print(f"Target Labels Map (Implicit Alphabetical Indexing):")
for idx, name in enumerate(class_names):
    print(f"  Index {idx} -> {name}")
    
assert class_names == test_dataset.classes, "Train and Test directory subfolders must match exactly!"

## Load Model 

Initialize a lightweight MobileNet model architecture pre-trained on ImageNet.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze features backbone to safeguard existing edge-detection filters
for param in model.features.parameters():
    param.requires_grad = False

num_classes = len(class_names)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Only update parameters on the newly initialized classification layer
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

## Define Training

Define training functions for one step along with defining metrics for this model.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(dataloader, desc="  Training Batches", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return running_loss / total, correct / total

def evaluate_blind(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    return running_loss / total, correct / total

## Train Model

Train the final classification layer to distinguish track structural shapes.

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": []
}

try:
    for epoch in range(EPOCHS):
        # Run execution iterations
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_blind(model, test_loader, criterion, device)
        
        # Record data metrics
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc * 100)
        history["test_loss"].append(val_loss)
        history["test_acc"].append(val_acc * 100)
        
        clear_output(wait=True) # Wipes the previous epoch's chart to prevent scrolling explosions
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), dpi=100)
        
        # Subplot A: Loss Convergences
        ax1.plot(history["epoch"], history["train_loss"], label="Train Loss", color="dodgerblue", marker='o')
        ax1.plot(history["epoch"], history["test_loss"], label="Blind Test Loss", color="crimson", marker='x', linestyle='--')
        ax1.set_title("Loss Tracking Metric Curves", fontsize=12, fontweight='bold')
        ax1.set_xlabel("Epoch Count")
        ax1.set_ylabel("Loss Magnitude")
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Subplot B: Accuracy Percentages
        ax2.plot(history["epoch"], history["train_acc"], label="Train Accuracy", color="dodgerblue", marker='o')
        ax2.plot(history["epoch"], history["test_acc"], label="Blind Test Accuracy", color="crimson", marker='x', linestyle='--')
        ax2.set_title("Accuracy Performance Curves", fontsize=12, fontweight='bold')
        ax2.set_xlabel("Epoch Count")
        ax2.set_ylabel("Accuracy Percentage (%)")
        ax2.set_ylim(0, 105) # Keeps the y-axis bounded nicely
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc="lower right")
        
        plt.suptitle(f"MobileNetV2 - Current Epoch: {epoch+1:02d}", fontsize=14, fontweight='bold', y=1.02)
        display(fig)
        plt.close(fig)
        
        # Print status indicators under the charts
        print(f"Latest Metrics -> Train Acc: {train_acc*100:.2f}% | Blind Test Acc: {val_acc*100:.2f}%")
        print("Press stop button to stop training.")

except KeyboardInterrupt:
    print("Your model state is preserved in memory at its current state.")

## Evaluate Network

Run blind validation checks across test data to compute accuracy benchmarks.

In [ ]:
model.eval()

num_classes = len(class_names)
matrix = np.zeros((num_classes, num_classes), dtype=int)

print("Gathering predictions for visual heatmap...")
with torch.no_grad():
	for images, labels in test_loader:
		images, labels = images.to(device), labels.to(device)
		outputs = model(images)
		_, predicted = outputs.max(1)
		
		for i in range(labels.size(0)):
			matrix[labels[i].item()][predicted[i].item()] += 1

# Initialize the Matplotlib Plot Space
fig, ax = plt.subplots(figsize=(10, 8), dpi=100)

# Render Heatmap with a clean, high-contrast color profile (Blues or Viridis)
im = ax.imshow(matrix, interpolation='nearest', cmap=plt.cm.Blues)
fig.colorbar(im, ax=ax, label="Sample Count")

# Configure Tick Marks and Axis Labels
tick_marks = np.arange(num_classes)
ax.set_xticks(tick_marks)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
ax.set_yticks(tick_marks)
ax.set_yticklabels(class_names, fontsize=10)

# Annotate every single grid box with its corresponding numerical integer count
thresh = matrix.max() / 2.
for i in range(num_classes):
	for j in range(num_classes):
		ax.text(j, i, format(matrix[i][j], 'd'),
				ha="center", va="center",
				color="white" if matrix[i][j] > thresh else "black",
				fontweight='bold', fontsize=12)
		
ax.set_title("Coaster Manufacturer Classification - Blind Test Matrix", fontsize=14, pad=20, fontweight='bold')
ax.set_xlabel("Predicted Manufacturer Label", fontsize=12, labelpad=10)
ax.set_ylabel("Actual Manufacturer Label", fontsize=12, labelpad=10)

plt.tight_layout()
plt.show()

# Export ONNX

Convert the trained PyTorch model weights into an optimized web-ready layout.

In [ ]:
MODEL_DIR = "../web/public/models/"

model.eval()

torch.onnx.export(
	model,
	torch.randn(1, 3, 224, 224, device=device),
	f"{MODEL_DIR}/track_classifier.onnx",
	export_params=True,
	opset_version=13,
	do_constant_folding=True,
	input_names=["input_tensor"],
	output_names=["logits_output"],
)

model_onnx = onnx.load(f"{MODEL_DIR}/track_classifier.onnx")
model_simp, check = simplify(model_onnx)

assert check, "Simplified ONNX model could not be validated"

onnx.save(model_simp, f"{MODEL_DIR}/track_classifier.onnx")

print("Saved and validated")